# 🌊 Early ENSO Phase Prediction — Starter Notebook

**Dataset:** [ENSO Early Phase Prediction](https://www.kaggle.com/datasets/ferariz/enso-early-phase-prediction)  
**Author:** ferariz  
**Last updated:** 2026

---

## What is ENSO?

The **El Niño–Southern Oscillation (ENSO)** is the dominant mode of interannual climate variability on Earth. It alternates between three phases:

| Phase | Niño 3.4 anomaly | Effect |
|---|---|---|
| **El Niño** | > +0.5 °C | Warm eastern Pacific, disrupted rainfall globally |
| **La Niña** | < −0.5 °C | Cool eastern Pacific, intensified trade winds |
| **Neutral** | −0.5 to +0.5 °C | Near-average conditions |

Predicting ENSO phase months in advance has direct consequences for drought forecasting, agriculture, and disaster preparedness across the tropics.

## Task

Given monthly climate indices observed at time **t**, predict the ENSO phase at:
- **t+1** (1 month ahead)
- **t+3** (3 months ahead)  
- **t+6** (6 months ahead)

This notebook covers:
1. [Data exploration](#1-data-exploration)
2. [Feature overview](#2-feature-overview)
3. [Simple baseline: Logistic Regression](#3-simple-baseline)
4. [Stronger model: LightGBM](#4-lightgbm)
5. [Feature importance (SHAP)](#5-feature-importance)
6. [Lead-time performance curve](#6-lead-time-curve)
7. [What to try next](#7-what-to-try-next)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import f1_score, accuracy_score, confusion_matrix
import lightgbm as lgb

plt.rcParams.update({
    'figure.dpi': 130,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.family': 'DejaVu Sans',
})

PHASE_COLORS = {'El Niño': '#d6604d', 'Neutral': '#999999', 'La Niña': '#4393c3'}
LABEL_ORDER  = ['La Niña', 'Neutral', 'El Niño']
print('Libraries loaded ✓')

In [ ]:
# ── Load data ─────────────────────────────────────────────────────────────────
train = pd.read_parquet('/kaggle/input/enso-early-phase-prediction/enso_train.parquet')
test  = pd.read_parquet('/kaggle/input/enso-early-phase-prediction/enso_test.parquet')

train['date'] = pd.to_datetime(train['date'])
test['date']  = pd.to_datetime(test['date'])

print(f'Train: {train.shape[0]} months ({train.date.min().date()} → {train.date.max().date()})')
print(f'Test:  {test.shape[0]} months  ({test.date.min().date()} → {test.date.max().date()})')
print(f'Columns: {train.shape[1]}')
train.head(3)

## 1. Data Exploration <a id='1-data-exploration'></a>

In [ ]:
# ── ENSO phase distribution ───────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))

for ax, target, title in zip(
    axes,
    ['enso_phase', 'enso_t3', 'enso_t6'],
    ['Phase at t (reference)', 'Target: t+3 months', 'Target: t+6 months']
):
    counts = train[target].value_counts().reindex(LABEL_ORDER)
    bars = ax.bar(
        LABEL_ORDER, counts,
        color=[PHASE_COLORS[p] for p in LABEL_ORDER],
        edgecolor='black', linewidth=0.5
    )
    ax.bar_label(bars, fmt='%d', padding=3, fontsize=9)
    ax.set_title(title, fontweight='bold', fontsize=10)
    ax.set_ylabel('Months')
    ax.set_ylim(0, counts.max() * 1.18)

plt.suptitle('Class distribution — training set', fontsize=12, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()
print('Note: Neutral is the most common class — use macro F1, not accuracy, as your primary metric.')

In [ ]:
# ── Niño 3.4 time series ──────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 4))

x = train['date']
y = train['sst_anom_nino34']

ax.axhline( 0.5, color='#d6604d', linestyle='--', linewidth=0.8, alpha=0.7, label='El Niño threshold (+0.5°C)')
ax.axhline(-0.5, color='#4393c3', linestyle='--', linewidth=0.8, alpha=0.7, label='La Niña threshold (−0.5°C)')
ax.axhline( 0.0, color='black',   linestyle='-',  linewidth=0.4, alpha=0.3)

ax.fill_between(x, y, 0, where=(y >  0.5), color='#d6604d', alpha=0.25)
ax.fill_between(x, y, 0, where=(y < -0.5), color='#4393c3', alpha=0.25)
ax.plot(x, y, color='black', linewidth=0.9)

# Annotate famous events
events = [
    ('1982-11', '1982/83\nEl Niño'),
    ('1997-11', '1997/98\nEl Niño'),
    ('2010-11', '2010/11\nLa Niña'),
    ('2015-11', '2015/16\nEl Niño'),
    ('2021-11', '2020–23\nLa Niña'),
]
for date_str, label in events:
    try:
        ts = pd.Timestamp(date_str)
        val = train.loc[train['date'] == ts, 'sst_anom_nino34'].values
        if len(val):
            ax.annotate(label, xy=(ts, val[0]), xytext=(0, 18 if val[0] > 0 else -28),
                       textcoords='offset points', fontsize=7.5, ha='center',
                       arrowprops=dict(arrowstyle='->', color='gray', lw=0.8))
    except:
        pass

ax.set_ylabel('SST anomaly (°C)', fontsize=11)
ax.set_title('Niño 3.4 SST anomaly — training period (1980–2018)', fontsize=12, fontweight='bold')
ax.legend(fontsize=9, frameon=False, loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
# ── Seasonal distribution of ENSO phases ─────────────────────────────────────
# Shows the spring predictability barrier: ENSO onset is harder to predict
# from boreal spring (Mar–May) than from any other season.

month_phase = train.copy()
month_phase['month'] = train['date'].dt.month

pivot = month_phase.groupby(['month', 'enso_phase']).size().unstack(fill_value=0)
pivot = pivot.reindex(columns=LABEL_ORDER)
pivot_pct = pivot.div(pivot.sum(axis=1), axis=0) * 100

month_names = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
pivot_pct.index = month_names

fig, ax = plt.subplots(figsize=(11, 4))
pivot_pct.plot(kind='bar', stacked=True, ax=ax,
               color=[PHASE_COLORS[p] for p in LABEL_ORDER],
               edgecolor='white', linewidth=0.3)
ax.set_xlabel('')
ax.set_ylabel('% of months', fontsize=10)
ax.set_title('ENSO phase by calendar month — spring predictability barrier visible in Mar–May',
             fontsize=10, fontweight='bold')
ax.legend(LABEL_ORDER, loc='upper right', fontsize=9, frameon=False)
ax.set_xticklabels(month_names, rotation=0)
plt.tight_layout()
plt.show()

## 2. Feature Overview <a id='2-feature-overview'></a>

The dataset provides 50 features, all **backward-looking** (no future leakage). For each of the 6 base climate indices, we have:

| Feature type | Example | Physical meaning |
|---|---|---|
| Current value | `sst_anom_nino34` | Instantaneous state |
| Lag 1/3/6 months | `sst_anom_nino34_lag3` | Memory at 1, 3, 6 month delays |
| 3-month rolling mean | `sst_anom_nino34_rm3` | Smoothed state ≈ ONI index |
| 3-month rolling std | `sst_anom_nino34_rstd3` | Is the signal strengthening? |
| Monthly diff | `sst_anom_nino34_diff1` | Rate of change |

Plus `month_sin` / `month_cos` to capture the seasonal cycle without discontinuity.

In [ ]:
# ── Feature–target correlations ───────────────────────────────────────────────
TARGET_COLS = {'enso_phase', 'enso_t1', 'enso_t3', 'enso_t6',
               'enso_t1_int', 'enso_t3_int', 'enso_t6_int', 'date'}

feature_cols = [c for c in train.columns if c not in TARGET_COLS]
numeric_features = train[feature_cols].select_dtypes(include='number')

fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharey=True)

for ax, target_int, title in zip(
    axes,
    ['enso_t1_int', 'enso_t3_int', 'enso_t6_int'],
    ['Correlation with t+1', 'Correlation with t+3', 'Correlation with t+6']
):
    corrs = numeric_features.corrwith(train[target_int]).abs().sort_values(ascending=False).head(15)
    colors = ['#d6604d' if 'nino34' in c else
              '#4393c3' if 'soi' in c else
              '#66c2a5' if 'wind' in c else '#aaaaaa'
              for c in corrs.index]
    corrs.sort_values().plot(kind='barh', ax=ax, color=colors, edgecolor='black', linewidth=0.3)
    ax.set_title(title, fontweight='bold', fontsize=10)
    ax.set_xlabel('|Pearson r|')
    ax.axvline(0.5, color='gray', linestyle='--', linewidth=0.7, alpha=0.5)

plt.suptitle('Top 15 features by correlation with target (|r|)', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()
print(f'Total features: {len(feature_cols)}')

## 3. Simple Baseline: Logistic Regression <a id='3-simple-baseline'></a>

We start with logistic regression on a small feature subset — the most interpretable possible model.
This establishes a floor for what machine learning can do beyond naive baselines.

In [ ]:
def evaluate(y_true, y_pred, name=''):
    """Compute accuracy and macro F1, print result."""
    mask  = pd.notna(y_true) & pd.notna(pd.Series(y_pred))
    yt    = np.array(y_true)[mask]
    yp    = np.array(y_pred)[mask]
    acc   = accuracy_score(yt, yp)
    f1    = f1_score(yt, yp, average='macro', zero_division=0, labels=LABEL_ORDER)
    print(f'  {name:35s}  acc={acc:.3f}  f1_macro={f1:.3f}  n={len(yt)}')
    return {'name': name, 'accuracy': acc, 'f1_macro': f1,
            'cm': confusion_matrix(yt, yp, labels=LABEL_ORDER)}


def prepare(df, feature_cols, target, dropna=True):
    """Return X, y with NaNs removed."""
    d = df[feature_cols + [target]].copy()
    if dropna:
        d = d.dropna()
    return d[feature_cols].fillna(0), d[target]


# ── Small interpretable feature set ──────────────────────────────────────────
CORE_FEATURES = [
    'sst_anom_nino34',
    'sst_anom_nino34_lag1',
    'sst_anom_nino34_lag3',
    'sst_anom_nino34_rm3',
    'southern_oscillation_index',
    'zonal_wind_850_anom',
    'month_sin',
    'month_cos',
]
CORE_FEATURES = [c for c in CORE_FEATURES if c in train.columns]

le = LabelEncoder().fit(LABEL_ORDER)
sc = StandardScaler()

lr_results = {}
print('Logistic Regression — core features only\n')
for target in ['enso_t1', 'enso_t3', 'enso_t6']:
    X_tr, y_tr = prepare(train, CORE_FEATURES, target)
    X_te, y_te = prepare(test,  CORE_FEATURES, target)

    X_tr_sc = sc.fit_transform(X_tr)
    X_te_sc = sc.transform(X_te)

    lr = LogisticRegression(C=1.0, max_iter=2000, class_weight='balanced', random_state=42)
    lr.fit(X_tr_sc, le.transform(y_tr))
    preds = le.inverse_transform(lr.predict(X_te_sc))
    lr_results[target] = evaluate(y_te, preds, name=f'LR core features / {target}')

print()
print('These numbers represent a reasonable lower bound.')
print('Using all 50 features or a tree model will push these higher.')

In [ ]:
# ── Confusion matrix for t+3 ──────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(5, 4))
cm = lr_results['enso_t3']['cm']
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True).clip(min=1)

sns.heatmap(cm_norm, annot=cm, fmt='d', cmap='RdBu_r', vmin=0, vmax=1,
            xticklabels=LABEL_ORDER, yticklabels=LABEL_ORDER,
            linewidths=0.5, ax=ax)
ax.set_xlabel('Predicted', fontsize=10)
ax.set_ylabel('Actual', fontsize=10)
ax.set_title('Logistic Regression — t+3 confusion matrix', fontsize=10, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Stronger Model: LightGBM <a id='4-lightgbm'></a>

Now we use all 50 features with LightGBM — the strongest tabular baseline.

In [ ]:
ALL_FEATURES = [c for c in train.columns if c not in TARGET_COLS
                and train[c].dtype != object]

lgb_results = {}
lgb_models  = {}

print('LightGBM — all features\n')
for target in ['enso_t1', 'enso_t3', 'enso_t6']:
    X_tr, y_tr = prepare(train, ALL_FEATURES, target)
    X_te, y_te = prepare(test,  ALL_FEATURES, target)

    model = lgb.LGBMClassifier(
        n_estimators=500, learning_rate=0.05, num_leaves=31,
        min_child_samples=20, class_weight='balanced',
        random_state=42, n_jobs=-1, verbosity=-1
    )
    model.fit(
        X_tr, le.transform(y_tr),
        eval_set=[(X_te, le.transform(y_te))],
        callbacks=[lgb.early_stopping(50, verbose=False)]
    )
    preds = le.inverse_transform(model.predict(X_te))
    lgb_results[target] = evaluate(y_te, preds, name=f'LightGBM all features / {target}')
    lgb_models[target]  = model

print()
print('Benchmark results from dataset authors (test set 2019–2026):')
print('  enso_t1: LightGBM F1=0.945  |  Persistence F1=0.858')
print('  enso_t3: LR        F1=0.790  |  Persistence F1=0.610')
print('  enso_t6: RF        F1=0.556  |  Persistence F1=0.419')

## 5. Feature Importance (SHAP) <a id='5-feature-importance'></a>

SHAP values tell us which features drive predictions — and whether the model has learned physically meaningful patterns.

In [ ]:
try:
    import shap
    shap_available = True
except ImportError:
    shap_available = False
    print('shap not installed — run: pip install shap')

if shap_available:
    X_tr_t3, y_tr_t3 = prepare(train, ALL_FEATURES, 'enso_t3')
    explainer   = shap.TreeExplainer(lgb_models['enso_t3'])
    shap_values = explainer.shap_values(X_tr_t3)

    # Mean |SHAP| per feature, averaged across classes
    mean_abs = np.abs(np.stack(shap_values)).mean(axis=(0, 1))
    shap_df  = pd.DataFrame({'feature': ALL_FEATURES, 'mean_abs_shap': mean_abs})
    shap_df  = shap_df.sort_values('mean_abs_shap', ascending=False).head(20)

    fig, ax = plt.subplots(figsize=(8, 7))
    colors = ['#d6604d' if 'nino34' in c else
              '#4393c3' if 'soi' in c else
              '#66c2a5' if 'wind' in c else
              '#f4a582' if 'nino' in c else '#cccccc'
              for c in shap_df['feature']]

    shap_df.sort_values('mean_abs_shap').plot(
        kind='barh', x='feature', y='mean_abs_shap',
        ax=ax, color=list(reversed(colors)),
        edgecolor='black', linewidth=0.3, legend=False
    )
    ax.set_xlabel('Mean |SHAP value|', fontsize=10)
    ax.set_title('Feature importance — LightGBM t+3 (SHAP)', fontsize=11, fontweight='bold')

    # Add a legend for colour coding
    from matplotlib.patches import Patch
    legend_elements = [
        Patch(facecolor='#d6604d', label='Niño 3.4'),
        Patch(facecolor='#4393c3', label='SOI'),
        Patch(facecolor='#66c2a5', label='Zonal wind'),
        Patch(facecolor='#f4a582', label='Other Niño'),
    ]
    ax.legend(handles=legend_elements, fontsize=8, frameon=False, loc='lower right')
    plt.tight_layout()
    plt.show()

    print('\nTop 5 most important features for t+3:')
    print(shap_df.head(5)[['feature', 'mean_abs_shap']].to_string(index=False))

## 6. Lead-Time Performance Curve <a id='6-lead-time-curve'></a>

How does prediction skill decay as we look further ahead? This is the central research question.

In [ ]:
# ── Collect all results ───────────────────────────────────────────────────────
# Baselines
persistence_f1 = {}
climatology_f1 = {}

for target in ['enso_t1', 'enso_t3', 'enso_t6']:
    X_te, y_te = prepare(test, ['enso_phase'], target, dropna=True)
    # Persistence: predict current phase
    pers = evaluate(y_te, X_te['enso_phase'].values, name=f'persistence/{target}')
    persistence_f1[target] = pers['f1_macro']
    # Climatology: always predict modal class from train
    modal = train['enso_phase'].value_counts().idxmax()
    clim  = evaluate(y_te, [modal] * len(y_te), name=f'climatology/{target}')
    climatology_f1[target] = clim['f1_macro']

lead_times = [1, 3, 6]
targets    = ['enso_t1', 'enso_t3', 'enso_t6']

lr_f1   = [lr_results[t]['f1_macro']  for t in targets]
lgb_f1  = [lgb_results[t]['f1_macro'] for t in targets]
pers_f1 = [persistence_f1[t]          for t in targets]
clim_f1 = [climatology_f1[t]          for t in targets]

# ── Plot ──────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 5))

ax.plot(lead_times, lgb_f1,  '-o', color='#FF5722', linewidth=2,   markersize=8,  label='LightGBM (all features)')
ax.plot(lead_times, lr_f1,   '-^', color='#2196F3', linewidth=2,   markersize=8,  label='Logistic Regression (core features)')
ax.plot(lead_times, pers_f1, ':s', color='#555555', linewidth=1.5, markersize=7,  label='Persistence baseline')
ax.plot(lead_times, clim_f1, ':D', color='#aaaaaa', linewidth=1.5, markersize=7,  label='Climatology baseline')

# Shade the "skill region" above persistence
ax.fill_between(lead_times, pers_f1, lgb_f1, alpha=0.08, color='#FF5722')

ax.set_xticks(lead_times)
ax.set_xticklabels(['t+1\n(1 month)', 't+3\n(3 months)', 't+6\n(6 months)'], fontsize=10)
ax.set_ylabel('F1 macro', fontsize=11)
ax.set_ylim(0, 1.05)
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.2f'))
ax.set_title('Prediction skill vs lead time — test set (2019–2026)',
             fontsize=12, fontweight='bold')
ax.legend(fontsize=9, frameon=False)

# Annotate spring predictability barrier note
ax.annotate(
    'Skill drops sharply\nat 6-month lead\n(spring barrier)',
    xy=(6, lgb_f1[2]), xytext=(5.2, lgb_f1[2] + 0.12),
    fontsize=8.5, color='#555555',
    arrowprops=dict(arrowstyle='->', color='#555555', lw=0.8)
)

plt.tight_layout()
plt.show()

## 7. What to Try Next <a id='7-what-to-try-next'></a>

The baselines above are deliberately simple. Here are directions worth exploring:

### Feature engineering
- **MJO features** — The Madden-Julian Oscillation is a known ENSO precursor at 30–90 day timescales. BOM publishes daily RMM indices.
- **Thermocline depth** — The depth of the 20°C isotherm in the equatorial Pacific (D20 index) captures subsurface heat content, a leading indicator of El Niño development.
- **Longer lags** — Try lags of 9 and 12 months to capture the annual cycle of ENSO recharge.

### Modelling
- **Temporal cross-validation** — Use walk-forward CV instead of a single split for more robust hyperparameter tuning.
- **Sequence models** — LSTM or Transformer on the raw monthly time series.
- **Probabilistic output** — Instead of a hard label, predict the probability of each phase. Useful for operational forecasting.
- **Ensemble** — Blend LightGBM and LR predictions, weighted by lead time.

### Evaluation
- **Heidke Skill Score (HSS)** — The standard meteorological metric for categorical ENSO forecasts. More interpretable than F1 for climate scientists.
- **El Niño onset detection** — Binary task: did an El Niño event start in the next 6 months?

---

**If you find something that beats the benchmarks, share it!**  
The t+6 result (F1≈0.55) has the most headroom — that's where the real challenge is.